# Dr Plant — Multi-crop PlantVillage Training v1

تدريب استكشافي متعدد المحاصيل (14 محصولاً / 38 فئة) باستخدام MobileNetV3-Small. لا يغيّر نموذج الليمون المحفوظ ولا يستبدله. النتائج ليست تحققاً حقلياً مستقلاً، والترخيص يحتاج مراجعة قبل التوزيع التجاري.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import os, json, sys, subprocess, shutil, csv, hashlib, time
DRIVE_ROOT = Path('/content/drive/MyDrive/DrPlant')
RUN_DIR = DRIVE_ROOT / 'multicrop-plantvillage-v1'
IMAGE_ROOT = Path('/content/plantvillage-images')
MANIFEST = Path('/content/plantvillage-manifest.csv')
RUN_DIR.mkdir(parents=True, exist_ok=True)
print('Drive mounted; output:', RUN_DIR)


In [ ]:
!pip -q install 'datasets>=3.0.0' scikit-learn pandas pillow
!git clone --branch training/multicrop-plantvillage-v1 https://github.com/jonbander/dr-plant.git /content/dr-plant
%cd /content/dr-plant
!git pull
print('Training script:', Path('scripts/train_image_classifier.py').exists())


## 1. تحميل مجموعة البيانات وفحص بنيتها
نستخدم النسخة المنظمة التي تعرض class_label وleaf_id وتقسيم train/test. بعض الفئات لها leaf IDs اصطناعية بحسب بطاقة البيانات؛ لذلك نضع النتيجة في خانة baseline استكشافي ولا ندّعي استقلالاً كاملاً لكل الصور.


In [ ]:
from datasets import load_dataset
ds = load_dataset('fatemehENG/plantvillage-full')
print(ds)
for split_name in ds:
    print(split_name, ds[split_name].column_names, len(ds[split_name]))
required = {'image','class_label','split','leaf_id'}
if not required.issubset(set(ds['train'].column_names)):
    raise RuntimeError(f'Dataset schema changed; expected {required}, got {ds["train"].column_names}')
classes = sorted(set(ds['train']['class_label']))
print('Classes:', len(classes))
print('\n'.join(classes))
if len(classes) != 38:
    raise RuntimeError(f'Expected 38 classes, found {len(classes)}; stop and review before training.')


## 2. تصدير الصور وإنشاء تقسيم يمنع تسرب مجموعات الأوراق قدر الإمكان
نستخدم تقسيم المصدر train/test، ونقتطع validation من train باستخدام StratifiedGroupKFold ومفتاح leaf_id. نوقف التنفيذ إذا لم تظهر كل فئة في الأقسام الثلاثة أو حدث تسرب للمجموعات.


In [ ]:
import numpy as np
from sklearn.model_selection import StratifiedGroupKFold
from collections import Counter
from PIL import Image
IMAGE_ROOT.mkdir(parents=True, exist_ok=True)
all_rows=[]
global_i=0
for source_split in ds:
    part=ds[source_split]
    for i, row in enumerate(part):
        label=str(row['class_label'])
        src_split=str(row['split']).lower()
        if src_split not in ('train','test'):
            raise RuntimeError(f'Unexpected source split: {src_split}')
        group=str(row['leaf_id']).strip()
        if not group:
            raise RuntimeError('Empty leaf_id; refusing split')
        out_dir=IMAGE_ROOT / label
        out_dir.mkdir(parents=True, exist_ok=True)
        filename=f'{global_i:06d}.jpg'
        rel=(Path(label)/filename).as_posix()
        out_path=IMAGE_ROOT/rel
        if not out_path.exists():
            im=row['image'].convert('RGB')
            im.save(out_path, format='JPEG', quality=95)
        all_rows.append({'relative_path':rel,'class_label':label,'source_split':src_split,'group_id':group})
        global_i+=1
    print('Exported', source_split, len(part))
print('Total exported:', len(all_rows), 'classes:', len(set(r['class_label'] for r in all_rows)))
train_rows=[r for r in all_rows if r['source_split']=='train']
test_rows=[r for r in all_rows if r['source_split']=='test']
y=np.array([r['class_label'] for r in train_rows])
groups=np.array([r['group_id'] for r in train_rows])
sgkf=StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
train_idx,val_idx=next(sgkf.split(np.zeros(len(y)), y, groups))
for idx in train_idx: train_rows[idx]['split']='train'
for idx in val_idx: train_rows[idx]['split']='validation'
for r in test_rows: r['split']='test'
manifest_rows=train_rows+test_rows
all_classes=set(r['class_label'] for r in manifest_rows)
for s in ('train','validation','test'):
    got=set(r['class_label'] for r in manifest_rows if r['split']==s)
    missing=sorted(all_classes-got)
    print(s, 'images=',sum(r['split']==s for r in manifest_rows),'classes=',len(got),'missing=',missing)
    if missing: raise RuntimeError(f'Classes missing from {s}; do not train')
group_sets={s:{r['group_id'] for r in manifest_rows if r['split']==s} for s in ('train','validation','test')}
if (group_sets['train']&group_sets['validation']) or (group_sets['train']&group_sets['test']) or (group_sets['validation']&group_sets['test']):
    raise RuntimeError('Group leakage detected')
with MANIFEST.open('w',newline='',encoding='utf-8') as f:
    fields=['relative_path','class_label','split','group_id_for_split','provenance_verified','expert_label_review','near_duplicate_review']
    w=csv.DictWriter(f,fieldnames=fields); w.writeheader()
    for r in manifest_rows:
        w.writerow({'relative_path':r['relative_path'],'class_label':r['class_label'],'split':r['split'],'group_id_for_split':r['group_id'],'provenance_verified':'yes','expert_label_review':'no','near_duplicate_review':'no'})
print('Manifest ready:', MANIFEST, 'rows:', len(manifest_rows))


## 3. التدريب
هذا تشغيل baseline استكشافي لأن التسميات لم تُراجع كلها بواسطة اختصاصي ولأن بعض معرفات الأوراق اصطناعية. لا تستخدم أرقام هذا التدريب كدقة ميدانية. إذا كان Colab على CPU فالأفضل إيقافه وتحويل Runtime إلى GPU قبل البدء.


In [ ]:
OUT = RUN_DIR / 'model-output'
cmd=[sys.executable,'scripts/train_image_classifier.py',str(IMAGE_ROOT),str(MANIFEST),'--out',str(OUT),'--epochs','8','--batch-size','16','--pretrained','--exploratory-non-independent']
print('Command:', ' '.join(cmd))
proc=subprocess.Popen(cmd,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
for line in proc.stdout:
    print(line,end='')
code=proc.wait()
if code != 0: raise RuntimeError(f'Training failed with exit code {code}; checkpoints, if any, remain in {OUT}')
print('Training completed.')


## 4. تصدير ONNX والتحقق
نصدّر أفضل checkpoint ونختبر أن ONNX Runtime يعطي مخرجات متقاربة مع PyTorch. لا ندمج النموذج في التطبيق قبل اختبار صور جديدة ومراجعة الترخيص.


In [ ]:
!pip -q install onnx onnxruntime
import torch, onnx, onnxruntime as ort
from torchvision import models
from torch import nn
import json, numpy as np
metrics=json.loads((OUT/'metrics.json').read_text(encoding='utf-8'))
labels=metrics['classes']
ck=torch.load(OUT/'best-model.pt',map_location='cpu',weights_only=False)
model=models.mobilenet_v3_small(weights=None)
model.classifier[3]=nn.Linear(model.classifier[3].in_features,len(labels))
model.load_state_dict(ck['state_dict']); model.eval()
dummy=torch.randn(1,3,224,224)
onnx_path=OUT/'plantvillage-multicrop-mobilenetv3-small.onnx'
torch.onnx.export(model,dummy,str(onnx_path),input_names=['input'],output_names=['logits'],dynamic_axes={'input':{0:'batch'},'logits':{0:'batch'}},opset_version=17)
onnx.checker.check_model(onnx.load(str(onnx_path)))
sess=ort.InferenceSession(str(onnx_path),providers=['CPUExecutionProvider'])
with torch.no_grad(): pt=model(dummy).numpy()
ox=sess.run(['logits'],{'input':dummy.numpy()})[0]
max_diff=float(np.max(np.abs(pt-ox)))
if max_diff>1e-3: raise RuntimeError(f'ONNX parity check failed: max diff={max_diff}')
(OUT/'labels.json').write_text(json.dumps(labels,ensure_ascii=False,indent=2)+'\n',encoding='utf-8')
summary={'dataset':'fatemehENG/plantvillage-full','images':len(manifest_rows),'classes':len(labels),'class_labels':labels,'split_counts':dict(Counter(r['split'] for r in manifest_rows)),'training_status':metrics.get('status'),'test_accuracy':metrics.get('test_accuracy'),'test_macro_f1':metrics.get('test_macro_f1'),'onnx_max_abs_difference':max_diff,'license_note':'Upstream PlantVillage is commonly marked CC BY-SA 3.0; verify original dataset terms and model derivative obligations before commercial distribution.','limitations':metrics.get('limitations',[])}
(RUN_DIR/'run-summary.json').write_text(json.dumps(summary,ensure_ascii=False,indent=2)+'\n',encoding='utf-8')
print(json.dumps(summary,ensure_ascii=False,indent=2))
print('ONNX:', onnx_path)


## 5. ما بعد التدريب
لا تعِد تشغيل التدريب لمجرد العودة إلى Colab. ملفات النموذج والنتائج محفوظة في Google Drive داخل `DrPlant/multicrop-plantvillage-v1/model-output`. الخطوة التالية بعد اكتمال التدريب هي اختبار خارجي على صور حقلية، ثم مراجعة توافق الترخيص قبل الدمج في التطبيق.
